# ML Threat Detection — Exploratory Data Analysis
## Assignment 04 | Network Intrusion Detection (CICIDS2017-style)

This notebook explores the network traffic dataset, visualizes class distribution and feature correlations, and shows training results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "network_traffic_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Class distribution — multi-class and binary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["label"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Multi-class Label Distribution")
axes[0].set_xlabel("Attack Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

df["label_binary"].value_counts().plot(kind="bar", ax=axes[1], color=["green", "red"])
axes[1].set_title("Binary Label Distribution (Benign vs Attack)")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("eda_class_distribution.png", dpi=150)
plt.show()

print("\nClass imbalance ratio (Benign / Total):", 
      f"{(df['label_binary']=='Benign').mean():.1%}")

In [ ]:
# Feature correlation heatmap (top 20 numeric features by variance)
feature_cols = [c for c in df.columns if c not in ("label", "label_binary")]
numeric_df = df[feature_cols].select_dtypes(include=[np.number])

top_var_cols = numeric_df.var().sort_values(ascending=False).head(20).index
corr = numeric_df[top_var_cols].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.5)
plt.title("Feature Correlation Heatmap (Top 20 by Variance)")
plt.tight_layout()
plt.savefig("eda_correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Data quality summary
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nNumeric features: {len(numeric_df.columns)}")
print(f"\nBasic stats for key features:")
df[["flow_duration", "total_packets", "total_bytes", "syn_flag_count"]].describe()

In [ ]:
# Training log and feature importance (run train_model.py first)
import joblib
from pathlib import Path

artifacts = Path("model_artifacts")
if (artifacts / "training_log.csv").exists():
    log = pd.read_csv(artifacts / "training_log.csv")
    display(log[["model", "accuracy", "precision", "recall", "f1", "roc_auc", "training_time_sec"]])

    # Feature importance from best Random Forest or XGBoost
    all_models = joblib.load(list(artifacts.glob("all_models_*.pkl"))[0])
    prep = joblib.load(artifacts / "preprocessing_pipeline.pkl")
    feature_names = prep["feature_names"]

    model = all_models.get("random_forest") or all_models.get("xgboost")
    if hasattr(model, "feature_importances_"):
        imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
        imp.plot(kind="barh", figsize=(10, 6), color="coral")
        plt.title("Top 15 Feature Importances")
        plt.xlabel("Importance")
        plt.tight_layout()
        plt.savefig("eda_feature_importance.png", dpi=150)
        plt.show()
else:
    print("Run 'python train_model.py' first to generate training artifacts.")

In [ ]:
# Neural Network training curve (loss vs epoch)
if artifacts.exists():
    all_models = joblib.load(list(artifacts.glob("all_models_*.pkl"))[0])
    nn = all_models.get("neural_network")
    if nn and hasattr(nn, "loss_curve_"):
        plt.figure(figsize=(8, 5))
        plt.plot(nn.loss_curve_, label="Training Loss")
        if hasattr(nn, "validation_scores_"):
            plt.plot(nn.validation_scores_, label="Validation Score")
        plt.xlabel("Epoch")
        plt.ylabel("Loss / Score")
        plt.title("Neural Network Training Curve")
        plt.legend()
        plt.tight_layout()
        plt.savefig("eda_nn_training_curve.png", dpi=150)
        plt.show()
    else:
        print("Neural network loss curve not available.")